# Lightweight OpenVINO Generation Demo with GPT-2

This notebook demonstrates how to export, quantize, and run fast LLM generation on CPU using OpenVINO and Hugging Face Optimum Intel.

### What you will learn:
- **Export & Quantization:** Export `gpt2` to OpenVINO format with 4-bit weight-only quantization.
- **Compilation:** Compile the OpenVINO model for CPU execution.
- **Prompt Lookahead Decoding (PLD):** Accelerate generation by looking up n-gram patterns in the input prompt.
- **Speculative Decoding:** Boost generation speed using a smaller assistant model (`distilgpt2`).

> **Note:** This lightweight demo is meant for fast local testing. Do not commit exported model cache directories to Git.


##  Instructions to Run the Notebook

###  Prerequisites & Dependencies
Make sure you have the required packages installed in your active Python environment:

```bash
pip install "optimum[openvino]" transformers torch nncf


###  How to Run
1. Open `quantized_generation_demo_gpt2.ipynb` in VS Code, Jupyter Notebook, or JupyterLab.
2. Select your Python kernel that has `optimum-intel` and `openvino` installed.
3. Click inside each code cell sequentially and press **`Shift + Enter`** to run.

---

###  What to Expect During Execution

* **First Run:**
  * Downloads the `gpt2` model weights (~500 MB) from Hugging Face Hub.
  * Quantizes model weights to 4-bit precision (takes ~5–10 seconds on CPU).
  * Creates a local directory named `gpt2_openvino/` containing `openvino_model.xml`, `openvino_model.bin`, and tokenizer configuration files.
  * Prints: `Model exported (if needed) and compiled. Saved at: gpt2_openvino`

* **Subsequent Runs:**
  * Automatically detects that `gpt2_openvino` already exists (`saved = True`).
  * Skips downloading and quantization completely, loading the compiled model directly from disk in under a second!


## Step 1: Export, Quantize (4-bit), and Compile GPT-2

In this step, we use **Optimum Intel** to export the Hugging Face `gpt2` model to OpenVINO format with **4-bit weight-only quantization**.

### What happens in this code:
1. **Quantization Setup (`OVWeightQuantizationConfig`)**: Sets up 4-bit weight-only quantization to dramatically reduce model size and accelerate inference on CPU.
2. **Export (`export=True`)**: Converts the Hugging Face model directly into OpenVINO Intermediate Representation (`openvino_model.xml` and `openvino_model.bin`).
3. **Stateless Mode (`stateful=False`)**: Keeps KV-cache stateless so the model remains compatible with speculative decoding.
4. **Local Saving & Caching**: Saves the exported model and tokenizer locally (`gpt2_openvino/`). If already exported, it loads directly from disk.
5. **Compilation (`model.compile()`)**: Prepares the OpenVINO execution engine for runtime inference on CPU.


In [ ]:
# Lightweight export + quantize + compile example (gpt2)
import os
from transformers import AutoTokenizer
from optimum.intel import OVModelForCausalLM, OVWeightQuantizationConfig

model_name = "gpt2"                  # target model (small)
save_name = model_name.replace("/", "_") + "_openvino"
device = "cpu"                       # change to "gpu" if you have OpenVINO GPU drivers
precision = "f32"

# weight-only quantization settings (small, safe)
quantization_config = OVWeightQuantizationConfig(bits=4, sym=False, group_size=128, ratio=0.8)

load_kwargs = {
    "device": device,
    "ov_config": {
        "PERFORMANCE_HINT": "LATENCY",
        "INFERENCE_PRECISION_HINT": precision,
        "CACHE_DIR": os.path.join(save_name, "model_cache"),
    },
    "compile": False,
    "quantization_config": quantization_config,
}

saved = os.path.exists(save_name)

# Export/quantize model (only if not saved). Keep export quick by using a small model.
model = OVModelForCausalLM.from_pretrained(
    model_name if not saved else save_name,
    export=not saved,
    stateful=False,   # use stateless export (needed for speculative decoding examples). Set to True for stateful usage.
    **load_kwargs,
)

tokenizer = AutoTokenizer.from_pretrained(model_name if not saved else save_name)

if not saved:
    # Save metadata only (do NOT commit these files)
    model.save_pretrained(save_name)
    tokenizer.save_pretrained(save_name)

# Compile model for runtime
model.compile()
print("Model exported (if needed) and compiled. Saved at:", save_name)

## Step 2: Compare Generation Strategies (Plain, PLD, and Speculative Decoding)

This step demonstrates three different text generation approaches using the compiled OpenVINO model to showcase how performance and decoding speed can be improved.

---

### Breakdown of the Code Sections

#### 1. Input Prompt Setup
* Tokenizes a sample Python code prompt (`def add(a, b)...`) into PyTorch tensors for model generation.

#### 2. Standard (Plain) Generation
* Runs basic autoregressive text generation using standard `model.generate()`.
* Serves as the baseline output for comparison.

#### 3. Prompt Lookahead Decoding (PLD)
* Configures `prompt_lookup_num_tokens=3`.
* **What is PLD?** PLD is a fast, draft-model-free technique that searches the input prompt for repeated n-gram patterns to predict candidate future tokens. It accelerates generation without extra memory or assistant models.

#### 4. Speculative Decoding (Draft-Assistant Model)
* Loads a smaller assistant model (`distilgpt2`) that shares the exact same tokenizer as the target model (`gpt2`).
* Exports the assistant model statelessly (`stateful=False`).
* **How it works:** 
  1. The small assistant model generates candidate tokens quickly (`num_assistant_tokens = 3`).
  2. The target 4-bit OpenVINO `gpt2` model verifies all candidate tokens in parallel in a single forward pass, significantly increasing generation speed!


In [ ]:
# Simple generation + PLD + optional speculative decoding
from transformers import TextStreamer

prompt = "def add(a, b):\n    \"\"\"Return the sum of two numbers.\"\"\"\n    "
inputs = tokenizer(prompt, return_tensors="pt")

# Plain generation
out = model.generate(
    **inputs,
    max_new_tokens=64,
    pad_token_id=tokenizer.eos_token_id,
)
print("Plain generation:\n", tokenizer.decode(out[0], skip_special_tokens=True))

# Prompt Lookahead Decoding (PLD)
out_pld = model.generate(
    **inputs,
    max_new_tokens=64,
    pad_token_id=tokenizer.eos_token_id,
    prompt_lookup_num_tokens=3,  # small value for demo
)
print("\nPLD generation:\n", tokenizer.decode(out_pld[0], skip_special_tokens=True))

# Optional: Speculative Decoding example (assistant must share tokenizer)
# Here we use distilgpt2 as a smaller assistant; they share the GPT-2 tokenizer.
# Export and compile assistant model statelessly first (if not already done).
assistant_name = "distilgpt2"
assistant_save = assistant_name.replace("/", "_") + "_openvino_stateless"
if not os.path.exists(assistant_save):
    asst_model = OVModelForCausalLM.from_pretrained(
        assistant_name,
        export=True,
        stateful=False,
        device=device,
        ov_config={"CACHE_DIR": os.path.join(assistant_save, "model_cache")},
        compile=False,
    )
    asst_model.save_pretrained(assistant_save)
else:
    asst_model = OVModelForCausalLM.from_pretrained(assistant_save)

asst_model.generation_config.num_assistant_tokens = 3
asst_model.generation_config.num_assistant_tokens_schedule = "const"
asst_model.compile()

# Run speculative decoding (target model must be stateless as above)
out_spec = model.generate(
    **inputs,
    max_new_tokens=64,
    pad_token_id=tokenizer.eos_token_id,
    assistant_model=asst_model,
)
print("\nSpeculative decoding generation:\n", tokenizer.decode(out_spec[0], skip_special_tokens=True))

##  Important Notes & Key Takeaways

*  Lightweight Demo Design:** This notebook uses `gpt2` (~124M parameters) to ensure fast export, quantization, and execution on any standard CPU without heavy resource usage.

* Stateless vs. Stateful Export (`stateful=False`):**
  * We use `stateful=False` because speculative decoding requires stateless tensor passing between the assistant and main models.
  * For standard single-model production applications, export with `stateful=True` into a separate directory for optimal KV-cache performance.

*  Assistant Model Tokenizer Matching:** The speculative decoding example uses `distilgpt2` as an assistant because it shares the exact same vocabulary as `gpt2`. If swapping assistant models, always ensure both models use identical tokenizers.

*  Hardware Target:** The default target device is set to `"cpu"`. If you have an integrated or discrete Intel GPU with OpenVINO drivers installed, you can simply change `device = "gpu"`.

*  Git Hygiene Reminder:** Exported OpenVINO IR files (`.xml`, `.bin`) and `model_cache/` folders are generated locally. Do **not** commit these generated directories to Git repository branches.

